In [ ]:
# Gradient scan vs training AD scores

This notebook:

- Loads the trained FiLM-conditioned VAE.
- Recomputes AD scores on a sample of the *training* attention matrices.
- Loads `gradient_scan_results.jsonl` from the Modal scan.
- Produces comparison plots:
  - Histogram of training vs scan AD scores.
  - 2D histograms of AD vs |grad| (embedding and one-hot).
  - 2D histogram of |emb grad| vs |one-hot grad|.

> **Note:** You may need to adjust the paths for the VAE checkpoint and the training attention data depending on how you downloaded them from Modal.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# ------------------------------------------------------------------
# Paths – adjust as needed
# ------------------------------------------------------------------
PROJECT_ROOT = Path(".")

# JSONL produced by gradient_scan_modal.py (downloaded from Modal volume)
GRAD_SCAN_PATH = PROJECT_ROOT / "gradient_scan_results.jsonl"

# VAE checkpoint (download from Modal vae-checkpoints volume if needed)
# e.g. `modal volume get vae-checkpoints vae_latest.pt` in this directory
VAE_CHECKPOINT_PATH = PROJECT_ROOT / "vae_latest.pt"

# Directory containing training attention .npz files, downloaded from
# the `attention-output` volume (see collect_attns_modal.py)
# Example command:
#   modal volume get attention-output attention_data ./attention_data_modal
TRAIN_ATTNS_DIR = PROJECT_ROOT / "attention_data_modal"

# Model / VAE hyperparameters (must match training + gradient_scan_modal.py)
NUM_LAYERS = 61
MAX_SEQ_LEN = 128
VAE_LATENT_CHANNELS = 2
VAE_LATENT_SPATIAL = 2
VAE_COND_DIM = NUM_LAYERS
AD_METRIC = "k_sum"  # "sum", "max", or "k_sum"
TOP_K = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")
print(f"Grad scan path: {GRAD_SCAN_PATH}")
print(f"VAE checkpoint path: {VAE_CHECKPOINT_PATH}")
print(f"Training attns dir: {TRAIN_ATTNS_DIR}")

In [ ]:
# ------------------------------------------------------------------
# FiLM-conditioned VAE definition (must match training)
# ------------------------------------------------------------------

class FiLMLayer(nn.Module):
    def __init__(self, cond_dim: int, num_channels: int):
        super().__init__()
        self.fc = nn.Linear(cond_dim, num_channels * 2)

    def forward(self, x: torch.Tensor, cond_vec: torch.Tensor) -> torch.Tensor:
        # cond_vec: (batch, cond_dim)
        gamma_beta = self.fc(cond_vec)  # (batch, 2 * C)
        gamma, beta = gamma_beta.chunk(2, dim=-1)
        gamma = gamma.view(x.size(0), -1, 1, 1)
        beta = beta.view(x.size(0), -1, 1, 1)
        return gamma * x + beta


class FiLMConditionedVAE(nn.Module):
    def __init__(
        self,
        input_size: int = MAX_SEQ_LEN,
        latent_channels: int = VAE_LATENT_CHANNELS,
        latent_spatial: int = VAE_LATENT_SPATIAL,
        num_layers: int = NUM_LAYERS,
        cond_dim: int = VAE_COND_DIM,
    ):
        super().__init__()
        self.input_size = input_size
        self.latent_channels = latent_channels
        self.latent_spatial = latent_spatial
        self.num_layers = num_layers
        self.cond_dim = cond_dim

        # Encoder: (1, N, N) -> (16, latent_spatial, latent_spatial)
        self.enc_conv1 = nn.Conv2d(1, 4, kernel_size=4, stride=2, padding=1)  # 128 -> 64
        self.film1 = FiLMLayer(cond_dim, 4)
        self.enc_conv2 = nn.Conv2d(4, 8, kernel_size=4, stride=2, padding=1)  # 64 -> 32
        self.film2 = FiLMLayer(cond_dim, 8)
        self.enc_conv3 = nn.Conv2d(8, 16, kernel_size=4, stride=2, padding=1)  # 32 -> 16
        self.film3 = FiLMLayer(cond_dim, 16)
        self.adaptive_pool = nn.AdaptiveAvgPool2d(latent_spatial)

        flat_dim = 16 * latent_spatial * latent_spatial
        latent_dim = latent_channels * latent_spatial * latent_spatial
        self.fc_mu = nn.Linear(flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(flat_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, flat_dim)

        self._pre_pool_size = input_size // 8  # 128 -> 16

        # Decoder: (latent_dim) -> (1, N, N)
        self.dec_upsample1 = nn.Upsample(size=self._pre_pool_size, mode="bilinear", align_corners=False)
        self.dec_film1 = FiLMLayer(cond_dim, 16)
        self.dec_convt1 = nn.ConvTranspose2d(16, 8, kernel_size=4, stride=2, padding=1)  # 16 -> 32
        self.dec_film2 = FiLMLayer(cond_dim, 8)
        self.dec_convt2 = nn.ConvTranspose2d(8, 4, kernel_size=4, stride=2, padding=1)   # 32 -> 64
        self.dec_film3 = FiLMLayer(cond_dim, 4)
        self.dec_convt3 = nn.ConvTranspose2d(4, 1, kernel_size=4, stride=2, padding=1)   # 64 -> 128

    def _encode(self, x, cond_vec):
        h = F.relu(self.film1(self.enc_conv1(x), cond_vec))
        h = F.relu(self.film2(self.enc_conv2(h), cond_vec))
        h = F.relu(self.film3(self.enc_conv3(h), cond_vec))
        h = self.adaptive_pool(h)
        h = h.flatten(1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    @staticmethod
    def _reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def _decode(self, z, cond_vec):
        h = self.fc_decode(z)
        h = h.view(-1, 16, self.latent_spatial, self.latent_spatial)
        h = self.dec_upsample1(h)
        h = F.relu(self.dec_film1(self.dec_convt1(h), cond_vec))
        h = F.relu(self.dec_film2(self.dec_convt2(h), cond_vec))
        h = F.relu(self.dec_film3(self.dec_convt3(h), cond_vec))
        return h

    def forward(self, x, layer_indices_one_hot):
        # x: (num_layers, 1, H, W) or (B, 1, H, W)
        # layer_indices_one_hot: (num_layers, cond_dim)
        mu, logvar = self._encode(x, layer_indices_one_hot)
        z = self._reparameterize(mu, logvar)
        recon = self._decode(z, layer_indices_one_hot)
        return recon, mu, logvar


# ------------------------------------------------------------------
# Load VAE
# ------------------------------------------------------------------

vae = FiLMConditionedVAE(
    num_layers=NUM_LAYERS,
    latent_channels=VAE_LATENT_CHANNELS,
    latent_spatial=VAE_LATENT_SPATIAL,
    cond_dim=VAE_COND_DIM,
).to(DEVICE).float()

if VAE_CHECKPOINT_PATH.exists():
    ckpt = torch.load(VAE_CHECKPOINT_PATH, map_location=DEVICE)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        vae.load_state_dict(ckpt["model_state_dict"])
    else:
        vae.load_state_dict(ckpt)
    print("Loaded VAE checkpoint.")
else:
    print(f"WARNING: VAE checkpoint not found at {VAE_CHECKPOINT_PATH}. Using random weights.")

vae.eval()
for p in vae.parameters():
    p.requires_grad_(False)

# Precompute layer one-hot conditioning vector
layer_indices = torch.arange(NUM_LAYERS, device=DEVICE)
layer_onehot = F.one_hot(layer_indices, num_classes=VAE_COND_DIM).float()

In [ ]:
# ------------------------------------------------------------------
# Helper: compute AD score for a batch of head-averaged attentions
# ------------------------------------------------------------------

@torch.no_grad()
def compute_ad_scores_for_attns(attn_list, max_samples=None):
    """Compute AD scores for a list of attention arrays.

    Each element of `attn_list` should be of shape (NUM_LAYERS, seq_len, seq_len),
    containing head-averaged attention for each layer. This matches how
    `gradient_scan_modal.py` feeds the VAE (after padding).
    """
    scores = []

    if max_samples is not None:
        attn_list = attn_list[:max_samples]

    for idx, attn in enumerate(attn_list):
        attn = torch.tensor(attn, dtype=torch.float32, device=DEVICE)  # (L, S, S)
        L, S, _ = attn.shape
        assert L == NUM_LAYERS, f"Expected {NUM_LAYERS} layers, got {L}"

        # Pad to MAX_SEQ_LEN x MAX_SEQ_LEN (same as gradient_scan_modal.py)
        padded = torch.zeros(NUM_LAYERS, 1, MAX_SEQ_LEN, MAX_SEQ_LEN,
                             device=DEVICE, dtype=torch.float32)
        padded[:, :, :S, :S] = attn.unsqueeze(1)

        recon, mu, logvar = vae(padded, layer_onehot)

        mask = torch.zeros(1, 1, MAX_SEQ_LEN, MAX_SEQ_LEN, device=DEVICE)
        mask[:, :, :S, :S] = 1.0
        n_pixels = mask.sum()

        # recon, padded: (L, 1, H, W)
        per_layer_mse = ((recon - padded) ** 2 * mask).sum(dim=(1, 2, 3)) / n_pixels

        if AD_METRIC == "sum":
            ad_score = per_layer_mse.sum().item()
        elif AD_METRIC == "max":
            ad_score = per_layer_mse.max().item()
        elif AD_METRIC == "k_sum":
            top_k_vals, _ = per_layer_mse.topk(min(TOP_K, NUM_LAYERS))
            ad_score = top_k_vals.sum().item()
        else:
            raise ValueError(f"Unknown AD_METRIC: {AD_METRIC}")

        scores.append(ad_score)

        if (idx + 1) % 100 == 0:
            print(f"  processed {idx + 1}/{len(attn_list)} training samples")

    return np.array(scores, dtype=np.float32)


# ------------------------------------------------------------------
# Load a subset of training attention matrices
# ------------------------------------------------------------------

train_attn_arrays = []

if TRAIN_ATTNS_DIR.exists():
    # Example convention: each file is an .npz with key 'attn'
    # and shape (NUM_LAYERS, seq_len, seq_len). Adjust if your
    # actual files differ.
    files = sorted(p for p in TRAIN_ATTNS_DIR.glob("**/*.npz"))
    print(f"Found {len(files)} training attention files.")

    MAX_TRAIN_FILES = 2000  # to keep things quick; increase if desired
    for i, path in enumerate(files[:MAX_TRAIN_FILES]):
        data = np.load(path)
        if "attn" in data:
            arr = data["attn"]
        elif "layer_attns" in data:
            arr = data["layer_attns"]
        else:
            # Fallback: assume single array
            arr = data[list(data.files)[0]]
        train_attn_arrays.append(arr)
    print(f"Loaded {len(train_attn_arrays)} attention arrays for training AD computation.")
else:
    print(f"WARNING: TRAIN_ATTNS_DIR {TRAIN_ATTNS_DIR} does not exist. "
          "Training AD histogram will be empty unless you set this correctly.")

if train_attn_arrays:
    train_ad_scores = compute_ad_scores_for_attns(train_attn_arrays)
    print(f"Computed {len(train_ad_scores)} training AD scores.")
else:
    train_ad_scores = np.array([])

In [ ]:
# ------------------------------------------------------------------
# Load gradient-scan results JSONL
# ------------------------------------------------------------------

grad_scan_scores = []
emb_grad_norms = []
onehot_grad_norms = []

if not GRAD_SCAN_PATH.exists():
    raise FileNotFoundError(f"Grad scan file not found: {GRAD_SCAN_PATH}")

with open(GRAD_SCAN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue

        if "ad_score" not in rec:
            continue  # skip pure error lines

        grad_scan_scores.append(rec.get("ad_score", 0.0))
        emb_grad_norms.append(rec.get("prefix_emb_grad_norm", 0.0))
        onehot_grad_norms.append(rec.get("prefix_onehot_grad_norm", 0.0))

grad_scan_scores = np.array(grad_scan_scores, dtype=np.float32)
emb_grad_norms = np.array(emb_grad_norms, dtype=np.float32)
onehot_grad_norms = np.array(onehot_grad_norms, dtype=np.float32)

print(f"Loaded {len(grad_scan_scores)} gradient-scan results.")
print(f"  Nonzero |emb grad| count: {(emb_grad_norms > 0).sum()} ")
print(f"  Nonzero |onehot grad| count: {(onehot_grad_norms > 0).sum()} ")

In [ ]:
# ------------------------------------------------------------------
# Plots: histograms and 2D histograms
# ------------------------------------------------------------------

# 1) Histogram: training vs scan AD scores
plt.figure(figsize=(8, 5))

if train_ad_scores.size > 0:
    sns.histplot(train_ad_scores, bins=50, color="tab:blue", stat="density",
                 label="Training AD", alpha=0.4)

sns.histplot(grad_scan_scores, bins=50, color="tab:orange", stat="density",
             label="Grad-scan AD", alpha=0.4)

plt.xlabel("AD score")
plt.ylabel("Density")
plt.title("AD scores: training vs gradient scan")
plt.legend()
plt.tight_layout()
plt.show()


# 2) 2D hist: AD vs |emb grad|
plt.figure(figsize=(7, 5))
plt.hist2d(grad_scan_scores, emb_grad_norms,
           bins=80, cmap="viridis", norm=None)
plt.colorbar(label="count")
plt.xlabel("AD score")
plt.ylabel("|embedding grad| (prefix)")
plt.title("AD vs |emb grad|")
plt.tight_layout()
plt.show()


# 3) 2D hist: AD vs |one-hot grad|
plt.figure(figsize=(7, 5))
plt.hist2d(grad_scan_scores, onehot_grad_norms,
           bins=80, cmap="magma", norm=None)
plt.colorbar(label="count")
plt.xlabel("AD score")
plt.ylabel("|one-hot grad| (prefix)")
plt.title("AD vs |one-hot grad|")
plt.tight_layout()
plt.show()


# 4) 2D hist: |emb grad| vs |one-hot grad|
plt.figure(figsize=(7, 5))
plt.hist2d(emb_grad_norms, onehot_grad_norms,
           bins=80, cmap="plasma", norm=None)
plt.colorbar(label="count")
plt.xlabel("|embedding grad| (prefix)")
plt.ylabel("|one-hot grad| (prefix)")
plt.title("|emb grad| vs |one-hot grad|")
plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------
# Extra sanity checks / studies
# ------------------------------------------------------------------

# Correlation coefficients (Pearson) between AD and gradient norms
from scipy.stats import pearsonr

mask_nonzero_emb = emb_grad_norms > 0
mask_nonzero_oh = onehot_grad_norms > 0

if mask_nonzero_emb.any():
    r_emb, p_emb = pearsonr(grad_scan_scores[mask_nonzero_emb],
                            emb_grad_norms[mask_nonzero_emb])
    print(f"Pearson corr(AD, |emb grad|) over nonzero grads: r={r_emb:.3f}, p={p_emb:.1e}")
else:
    print("No nonzero embedding gradients to correlate.")

if mask_nonzero_oh.any():
    r_oh, p_oh = pearsonr(grad_scan_scores[mask_nonzero_oh],
                          onehot_grad_norms[mask_nonzero_oh])
    print(f"Pearson corr(AD, |one-hot grad|) over nonzero grads: r={r_oh:.3f}, p={p_oh:.1e}")
else:
    print("No nonzero one-hot gradients to correlate.")

if mask_nonzero_emb.any() and mask_nonzero_oh.any():
    mask_both = mask_nonzero_emb & mask_nonzero_oh
    if mask_both.any():
        r_both, p_both = pearsonr(emb_grad_norms[mask_both],
                                  onehot_grad_norms[mask_both])
        print(f"Pearson corr(|emb grad|, |one-hot grad|): r={r_both:.3f}, p={p_both:.1e}")

# Inspect top-k most anomalous scan prefixes
K = 10
idx_sorted = np.argsort(-grad_scan_scores)[:K]
print(f"\nTop {K} gradient-scan AD scores:")

with open(GRAD_SCAN_PATH, "r", encoding="utf-8") as f:
    all_records = [json.loads(line) for line in f if line.strip()]

for rank, idx in enumerate(idx_sorted, start=1):
    rec = all_records[idx]
    print(f"[{rank}] AD={grad_scan_scores[idx]:.6f} |emb|={emb_grad_norms[idx]:.2e} "
          f"|onehot|={onehot_grad_norms[idx]:.2e} type={rec.get('prefix_type')}\n"
          f"    prefix: {rec.get('prefix_text', '')[:120]}")